In [ ]:
import pandas as pd
import numpy as np

market_data = pd.read_csv(r"market_data.csv").drop(columns='Unnamed: 0')

In [2]:
market_data = market_data.sort_values(by=['asset', 'date'])
market_data ['date'] = pd.to_datetime(market_data['date'])
market_data = market_data
market_data

,date,asset,open,high,low,close,adj_close,volume,asset_class,ccy,fx_rate,spread,bid,ask,source,week_day,business_day
0,2022-01-03,AAPL,177.830002,182.880005,177.710007,182.009995,178.103638,104487900.0,Equity,USD,1.0,0.27301,181.873490,182.146500,Bloomberg,Monday,True
1,2022-01-04,AAPL,182.630005,182.940002,179.119995,179.699997,175.843216,99310400.0,Equity,USD,1.0,0.26955,179.565222,179.834772,Bloomberg,Tuesday,True
2,2022-01-05,AAPL,179.610001,180.169998,174.639999,174.919998,171.165787,94537600.0,Equity,USD,1.0,0.26238,174.788808,175.051188,Bloomberg,Wednesday,True
3,2022-01-06,AAPL,172.699997,175.300003,171.639999,172.000000,168.308533,96904000.0,Equity,USD,1.0,0.25800,171.871000,172.129000,Bloomberg,Thursday,True
4,2022-01-07,AAPL,172.889999,174.139999,171.029999,172.169998,168.474869,86709100.0,Equity,USD,1.0,0.25825,172.040873,172.299123,Bloomberg,Friday,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5500,2026-03-23,TLT,86.180000,86.720001,85.940002,86.389999,86.389999,69686700.0,Fixed Income,USD,1.0,0.06911,86.355444,86.424554,Reuters,Monday,True
5501,2026-03-24,TLT,85.720001,86.290001,85.559998,86.010002,86.010002,51799400.0,Fixed Income,USD,1.0,0.06881,85.975597,86.044407,Reuters,Tuesday,True
5502,2026-03-25,TLT,86.750000,86.879997,86.480003,86.839996,86.839996,37861000.0,Fixed Income,USD,1.0,0.06947,86.805261,86.874731,Reuters,Wednesday,True
5503,2026-03-26,TLT,86.339996,86.610001,85.930000,86.110001,86.110001,39537000.0,Fixed Income,USD,1.0,0.06889,86.075556,86.144446,Reuters,Thursday,True


In [3]:
from src_functions import completeness

completeness(market_data)

,date,close,ccy,source,spread,fx_rate,volume,bid,ask
asset,,,,,,,,,
AAPL,1.0,0.964578,1.0,1.0,1.0,1.0,1.0,1.0,1.0
EURUSD=X,1.0,0.999092,1.0,1.0,1.0,1.0,NaN,1.0,1.0
GLD,1.0,0.964578,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SPY,1.0,0.964578,1.0,1.0,1.0,1.0,1.0,1.0,1.0
TLT,1.0,0.964578,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### Data Validation

After checking data completeness, the next step is to assess whether populated values are logically valid.

The purpose is not to test market realism in a complex way, but to confirm that available observations satisfy basic consistency rules. Missing values are not treated as validation failures here, since they are already captured in the completeness section.

In [4]:
from src_functions import flag_validation

validation_df = flag_validation(market_data)

In [5]:
from src_functions import data_validation

validation_heatmap = data_validation(validation_df)
validation_heatmap

,close_valid,volume_valid,bid_valid,ask_valid,spread_valid
asset,,,,,
AAPL,100.0,100.0,100.0,100.0,100.0
EURUSD=X,100.0,100.0,100.0,100.0,100.0
GLD,100.0,100.0,100.0,100.0,100.0
SPY,100.0,100.0,100.0,100.0,100.0
TLT,100.0,100.0,100.0,100.0,100.0


### Outlier Detection

To complement rule-based validation, I added an outlier detection layer based on daily log returns.

Outliers are identified using a **3-sigma approach at asset level**. For each asset, I compute the mean and standard deviation of daily log returns and define the expected range as:

- **Upper bound = mean + 3 × standard deviation**
- **Lower bound = mean - 3 × standard deviation**

Any return outside this range is flagged as an outlier.

This approach is simple and interpretable, but it should not be read mechanically. A flagged observation is not automatically a data error: it may also reflect a genuine market shock or an unusually volatile trading day.

I also track negative outliers separately, since extreme downside moves are more relevant from a risk management perspective than positive price jumps.

In [6]:
from src_functions import outlier_and_returns

outliers_table = outlier_and_returns(market_data)
outliers_table

,tot_outliers_count,neg_outliers_count,neg_outliers_pct_on_tot,neg_outliers_pct_on_tot_outliers
asset,,,,
AAPL,13,6,0.544959,46.153846
EURUSD=X,11,4,0.363306,36.363636
GLD,13,9,0.817439,69.230769
SPY,12,10,0.908265,83.333333
TLT,7,3,0.272480,42.857143


- `neg_outliers_pct_on_tot` measures the share of total observations flagged as negative outliers.

- `neg_outliers_pct_on_tot_outliers` measures the share of negative outliers within the full set of outliers.

Negative outliers represent a very small fraction of total observations (below 1%), confirming that extreme downside events are relatively rare over the sample period.